In [ ]:
PROJECT_ID = "ghp-poc"
REGION = "us-central1"

### Dependency Imports

In [ ]:
from typing import Optional

import pydantic
import pandas as pd
from thefuzz import fuzz
from google import genai
from google.genai import types as genai_types
from google.cloud import bigquery

from census import Census
from us import states
import requests

import warnings
warnings.filterwarnings("ignore")

### Initialize Clients

In [ ]:
bq_client = bigquery.Client(project=PROJECT_ID)
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

In [ ]:
CENSUS_API_KEY = ""
c = Census(CENSUS_API_KEY)

In [ ]:
c.acs.get(('NAME', 'B25034_010E'), {'for': 'state:%s' % states.MD.fips})

In [ ]:
# Function to get city population
def get_city_population(city_name:str,
                         state_name:str) -> float:
    """Fetches the population for the given city_name and state_name from census API

    Args:
        city_name (str): The name of the city.
        state_name (str): The name of the state.
        If state_name is not mentioned specifically in the user query then infer it from knowledge

    Returns:
        float: population
    """
    print(f"called get_city_population with city_name={city_name}, state_name={state_name}")
    state_fips = get_state_fips(state_name)
    if not state_fips:
        print(f"State '{state_name}' not found.")
        return

    place_fips = get_place_fips(city_name, state_fips)
    if not place_fips:
        print(f"City '{city_name}' not found in {state_name}.")
        return

    # Fetch population
    population_variable = "B01003_001E"  # Total population
    result = c.acs1.get((population_variable, "NAME"), {'for': f'place:{place_fips}', 'in': f'state:{state_fips}'})

    if result:
        city_full_name = result[0]["NAME"]
        population = result[0][population_variable]
        # print(f"Population of {city_full_name}: {population}")
        return population
    else:
        return 0
        print("No data found.")

In [ ]:
def get_state_fips(state_name):
    url = "https://api.census.gov/data/2021/pep/population?get=NAME,STATE&for=state:*"
    response = requests.get(url)
    data = response.json()

    for row in data[1:]:  # Skip header row
        if row[0].lower() == state_name.lower():
            return row[1]  # Return the state FIPS code
    return None

def get_place_fips(city_name, state_fips):
    places = c.acs1.get(["NAME"], {'for': 'place:*', 'in': f'state:{state_fips}'})

    for place in places:
        if city_name.lower() in place["NAME"].lower():
            return place["place"]
    return None

def get_city_statistics(city_name: str, state_name: str):
    """Fetches various statistics for the given city_name and state_name from Census API"""
    """
Median Age: "B01002_001E"
Total population 25+ yrs: "B15003_001E"
Percent Foreign Born: "B05012_003E" (Foreign-born population) / "B01003_001E" (Total population)
Percent Earned Bachelor's Degree or Higher (25 yrs and over), 2023: "B15003_022E", "B15003_023E", "B15003_024E", "B15003_025E" (Sum of these categories) / "B15003_001E"
Percent Earned Graduate or Professional Degree (25 yrs and over), 2023: "B15003_024E" (Master's), "B15003_025E" (Doctorate/Professional) / "B15003_001E"
"""

    print(f"Fetching data for {city_name}, {state_name}")

    state_fips = get_state_fips(state_name)
    if not state_fips:
        print(f"State '{state_name}' not found.")
        return

    place_fips = get_place_fips(city_name, state_fips)
    if not place_fips:
        print(f"City '{city_name}' not found in {state_name}.")
        return

    variables = {
        "median_age": "B01002_001E",
        "foreign_born": "B05012_003E",
        "total_population": "B01003_001E",
        "bachelors_higher": ["B15003_022E", "B15003_023E", "B15003_024E", "B15003_025E"],
        "graduate_degree": ["B15003_024E", "B15003_025E"],
        "total_25plus": "B15003_001E"
    }


    # Fetch data
    result = c.acs1.get(
        [variables["median_age"], variables["foreign_born"], variables["total_population"],
         *variables["bachelors_higher"], *variables["graduate_degree"], variables["total_25plus"]],
        {'for': f'place:{place_fips}', 'in': f'state:{state_fips}'}
    )

    if not result:
        print("No data found.")
        return {}

    data = result[0]

    total_population = float(data.get(variables["total_population"], 0))
    total_25plus = float(data.get(variables["total_25plus"], 0))

    bachelors_higher_count = sum(float(data.get(var, 0)) for var in variables["bachelors_higher"])
    graduate_degree_count = sum(float(data.get(var, 0)) for var in variables["graduate_degree"])
    foreign_born_count = float(data.get(variables["foreign_born"], 0))

    print(data)
    stats = {
        "City": city_name,
        "Total Population": total_population,
        "Population above 25 age": total_25plus,
        "Median Age": float(data.get(variables["median_age"], 0)),
        "Percent Foreign Born": (foreign_born_count / total_population) * 100 if total_population else 0,
        "Percent Earned Bachelor's Degree or Higher (25+)": (bachelors_higher_count / total_25plus) * 100 if total_25plus else 0,
        "Percent Earned Graduate or Professional Degree (25+)": (graduate_degree_count / total_25plus) * 100 if total_25plus else 0
    }

    return stats

In [ ]:
get_city_population("San Francisco", "California")

In [ ]:
get_city_statistics("San Francisco", "California")

## Run Agent

In [ ]:
system_instruction = """You help with fetching data from third-party APIs for the given user query."""

In [ ]:
class ChatAgent:
    def __init__(self, model_name):
        self.model_name = model_name
        self.chat = genai_client.aio.chats.create(
            model=self.model_name,
            config=genai_types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.05,
                candidate_count=1,
                seed=42,
                tools=[get_city_population],
            ),
        )

    async def ask(self, query):
        response = await self.chat.send_message(query)
        print(response.text)
        # return response

    def clear(self):
        self.chat = genai_client.aio.chats.create(
            model=self.model_name,
            config=genai_types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.05,
                candidate_count=1,
                seed=42,
                tools=[get_city_population],
            ),
        )

In [ ]:
chat_agent = ChatAgent("gemini-2.0-flash")

In [ ]:
chat_agent.clear()
await chat_agent.ask("what is the population for New York city?")

In [ ]:
chat_agent.clear()
await chat_agent.ask("what is the population for Houston, new york city?")